# GenreVision V4 — EfficientNet-B2
V3 ile aynı pipeline (veri, augmentation, split, pos_weight, threshold tuning).
Tek fark: ResNet18 → **EfficientNet-B2** backbone.

In [ ]:
!pip install -q boto3 iterative-stratification scikit-learn

In [ ]:
import os, time, json
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from PIL import Image
from tqdm.auto import tqdm
import boto3
from sklearn.metrics import f1_score, classification_report
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

from google.colab import drive
drive.mount('/content/drive')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# === KONFİGÜRASYON ===
ACCESS_KEY    = '26ff1aa1645b69cac212507ec4c079dd'
SECRET_KEY    = '37872cda19b45e33509d5b84a3514234a5a06755481363e1561cb7a404f23a77'
ENDPOINT_URL  = 'https://8a97a7792885ace66acb04fa79b9ded7.r2.cloudflarestorage.com'
BUCKET_NAME   = 'genrevision'

DRIVE_DIR  = '/content/drive/MyDrive/GenreVision'
CACHE_DIR  = '/content/posters'
os.makedirs(CACHE_DIR, exist_ok=True)

# V4 hiperparametreler
BATCH_SIZE     = 48       # EfficientNet-B2 biraz daha fazla VRAM kullanır; T4'te güvenli değer
EPOCHS_MAX     = 25
PATIENCE       = 5
LR             = 3e-4
WEIGHT_DECAY   = 5e-4
GRAD_CLIP      = 1.0
NUM_WORKERS    = 2

MIXUP_ALPHA    = 0.2
DROPOUT_P      = 0.4      # EfficientNet zaten kendi içinde dropout kullanır; 0.4 yeterli
POS_WEIGHT_CLIP = 25.0

assert os.path.exists(f'{DRIVE_DIR}/train_dataset_50k_clean.csv'), 'CSV bulunamadı!'
assert os.path.exists(f'{DRIVE_DIR}/pos_weight.pt'), 'pos_weight bulunamadı!'
print('✓ Konfig OK')

In [ ]:
# === CACHE KONTROLÜ — V3 ile aynı, poster klasörü doluysa atlar ===
df = pd.read_csv(f'{DRIVE_DIR}/train_dataset_50k_clean.csv')
print(f'Toplam afiş: {len(df):,}')

mevcut = sum(1 for fid in df['id']
             if os.path.exists(os.path.join(CACHE_DIR, f'{int(fid)}.jpg')))
print(f"Cache'de mevcut: {mevcut:,} / {len(df):,}")

if mevcut < len(df):
    print("Cache eksik, R2'den indiriliyor...")

    def make_s3():
        return boto3.client('s3', endpoint_url=ENDPOINT_URL,
            aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
            region_name='auto')

    def download_one(film_id):
        key  = f'{int(film_id)}.jpg'
        path = os.path.join(CACHE_DIR, key)
        if os.path.exists(path) and os.path.getsize(path) > 1000:
            return 'skip'
        try:
            make_s3().download_file(BUCKET_NAME, key, path)
            return 'ok'
        except Exception:
            return 'fail'

    sayac, fail_ids = {'ok': 0, 'skip': 0, 'fail': 0}, []
    with ThreadPoolExecutor(max_workers=24) as ex:
        futures = {ex.submit(download_one, fid): fid for fid in df['id']}
        for fut in tqdm(as_completed(futures), total=len(futures), desc='Cache'):
            r = fut.result()
            sayac[r] += 1
            if r == 'fail':
                fail_ids.append(futures[fut])
    print(sayac)
    if fail_ids:
        df = df[~df['id'].isin(fail_ids)].reset_index(drop=True)

print(f'\n✓ Eğitim için kullanılabilir: {len(df):,}')

In [ ]:
# === AUGMENTATION + DATASET + MIXUP ===

train_tf = transforms.Compose([
    transforms.Resize((260, 260)),        # EfficientNet-B2'nin önerilen giriş boyutu 260
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class PosterDataset(Dataset):
    def __init__(self, df, label_cols, cache_dir, transform):
        self.df        = df.reset_index(drop=True)
        self.label_cols = label_cols
        self.cache_dir = cache_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        film_id = int(self.df.iloc[idx]['id'])
        path    = os.path.join(self.cache_dir, f'{film_id}.jpg')
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224))
        img    = self.transform(img)
        labels = torch.tensor(
            self.df.iloc[idx][self.label_cols].values.astype('float32')
        )
        return img, labels

def mixup_data(x, y, alpha):
    if alpha <= 0:
        return x, y
    lam  = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[perm], lam * y + (1 - lam) * y[perm]

label_cols = sorted([c for c in df.columns if c.startswith('Tur_')])
print(f'Tür sayısı: {len(label_cols)}')
print(f'Augmentation: Resize(260)→RandomCrop(224) + RandAugment(2,9) + ColorJitter + Mixup(α={MIXUP_ALPHA})')

In [ ]:
# === TRAIN / VAL / TEST SPLIT ===
sp1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
idx_train, idx_temp = next(sp1.split(df, df[label_cols].values))
df_train = df.iloc[idx_train].reset_index(drop=True)
df_temp  = df.iloc[idx_temp].reset_index(drop=True)

sp2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
idx_val, idx_test = next(sp2.split(df_temp, df_temp[label_cols].values))
df_val  = df_temp.iloc[idx_val].reset_index(drop=True)
df_test = df_temp.iloc[idx_test].reset_index(drop=True)

print(f'Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}')
df_test.to_csv(f'{DRIVE_DIR}/test_split_v4.csv', index=False)

train_ds = PosterDataset(df_train, label_cols, CACHE_DIR, train_tf)
val_ds   = PosterDataset(df_val,   label_cols, CACHE_DIR, eval_tf)
test_ds  = PosterDataset(df_test,  label_cols, CACHE_DIR, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
print(f'Batch sayısı — train: {len(train_loader)}, val: {len(val_loader)}')

In [ ]:
# === MODEL — EfficientNet-B2 ===

model = efficientnet_b2(weights=EfficientNet_B2_Weights.DEFAULT)

# EfficientNet-B2'nin son katmanı model.classifier[1]'dir (in_features=1408)
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(in_features, len(label_cols))
)
model = model.to(device)

# Sınıf dengesizliğini düzeltmek için pos_weight (V3'teki aynı dosyadan)
pw = torch.load(f'{DRIVE_DIR}/pos_weight.pt')
assert pw['tur_kolonlari'] == label_cols
clipped_pw  = np.clip(pw['pos_weight'].numpy(), 1.0, POS_WEIGHT_CLIP)
pos_weight  = torch.tensor(clipped_pw, dtype=torch.float32, device=device)
print(f'pos_weight aralığı: {clipped_pw.min():.2f} - {clipped_pw.max():.2f}')

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)
scaler = GradScaler('cuda')

print(f'Backbone: EfficientNet-B2  |  Classifier in_features: {in_features}')
print(f'Toplam parametre: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# === EĞİTİM DÖNGÜSÜ (V4) ===

def evaluate(model, loader):
    model.eval()
    losses, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with autocast('cuda'):
                out  = model(x)
                loss = criterion(out, y)
            losses.append(loss.item())
            preds = (torch.sigmoid(out) > 0.5).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(y.cpu().numpy())
    preds  = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    return (
        float(np.mean(losses)),
        f1_score(labels, preds, average='macro', zero_division=0),
        f1_score(labels, preds, average='micro', zero_division=0)
    )

best_val_f1  = -1.0
patience_ctr = 0
history      = []

print(f'V4 eğitim başlıyor: max {EPOCHS_MAX} epoch, patience={PATIENCE}\n')

for epoch in range(1, EPOCHS_MAX + 1):
    model.train()
    train_losses = []
    t0   = time.time()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS_MAX}', leave=False)

    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        x_mix, y_mix = mixup_data(x, y, MIXUP_ALPHA)

        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            out  = model(x_mix)
            loss = criterion(out, y_mix)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        train_losses.append(loss.item())
        pbar.set_postfix(loss=f'{loss.item():.4f}')

        if not np.isfinite(loss.item()):
            raise RuntimeError(f'Epoch {epoch}: NaN loss!')

    train_loss                       = float(np.mean(train_losses))
    val_loss, val_f1_macro, val_f1_micro = evaluate(model, val_loader)
    scheduler.step(val_f1_macro)
    epoch_time = time.time() - t0
    lr_now     = optimizer.param_groups[0]['lr']

    print(f'Epoch {epoch:02d}: '
          f'train={train_loss:.4f} | val={val_loss:.4f} | '
          f'F1m={val_f1_macro:.3f} | F1µ={val_f1_micro:.3f} | '
          f'lr={lr_now:.5f} | {epoch_time:.0f}s')

    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'val_f1_macro': float(val_f1_macro), 'val_f1_micro': float(val_f1_micro),
        'lr': lr_now, 'epoch_time_sec': epoch_time
    })
    pd.DataFrame(history).to_csv(f'{DRIVE_DIR}/training_history_v4.csv', index=False)

    if val_f1_macro > best_val_f1:
        best_val_f1  = val_f1_macro
        patience_ctr = 0
        torch.save({
            'model_state':   model.state_dict(),
            'label_cols':    label_cols,
            'epoch':         epoch,
            'val_loss':      val_loss,
            'val_f1_macro':  float(val_f1_macro),
            'val_f1_micro':  float(val_f1_micro)
        }, f'{DRIVE_DIR}/genre_vision_v4_best.pth')
        print(f'  ✓ Best F1_macro={val_f1_macro:.3f} → genre_vision_v4_best.pth')
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'\nEarly stopping! {PATIENCE} epoch boyunca F1_macro yükselmedi.')
            break

print(f'\n=== V4 eğitim bitti ===')
print(f'En iyi val F1_macro: {best_val_f1:.3f}')

In [ ]:
# === KAYIP & F1 EĞRİLERİ ===
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(hist_df['epoch'], hist_df['train_loss'], label='Train', color='#2E86AB', marker='o', markersize=3)
ax1.plot(hist_df['epoch'], hist_df['val_loss'],   label='Val',   color='#F18805', marker='s', markersize=3)
ax1.set_title('Kayıp Eğrisi — EfficientNet-B2 (V4)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(hist_df['epoch'], hist_df['val_f1_macro'], label='F1 Macro', color='#A23E48', linewidth=2)
ax2.plot(hist_df['epoch'], hist_df['val_f1_micro'], label='F1 Micro', color='#6B4226', linewidth=2, linestyle='--')
ax2.set_title('Validation F1 Gelişimi — EfficientNet-B2 (V4)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.legend()

plt.tight_layout()
plt.savefig('v4_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === PER-CLASS THRESHOLD TUNING ===

ckpt = torch.load(f'{DRIVE_DIR}/genre_vision_v4_best.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

def get_probs_and_labels(model, loader):
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            with autocast('cuda'):
                out = model(x)
            all_probs.append(torch.sigmoid(out).cpu().numpy())
            all_labels.append(y.numpy())
    return np.vstack(all_probs), np.vstack(all_labels)

print('Validation üzerinde olasılıklar hesaplanıyor...')
val_probs, val_labels = get_probs_and_labels(model, val_loader)

best_thresholds = np.full(len(label_cols), 0.5)
print(f'\n{"Tür":<22} {"Eşik":>8} {"F1 (0.5)":>10} {"F1 (tuned)":>12}')
print('-' * 56)

for i, tur in enumerate(label_cols):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.10, 0.70, 0.02):
        preds = (val_probs[:, i] > t).astype(int)
        f1    = f1_score(val_labels[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[i] = best_t

    eski_f1  = f1_score(val_labels[:, i], (val_probs[:, i] > 0.5).astype(int), zero_division=0)
    fark     = best_f1 - eski_f1
    isaret   = f' (+{fark:.3f})' if fark > 0.005 else ''
    print(f'{tur.replace("Tur_",""):<22} {best_t:>8.2f} {eski_f1:>10.3f} {best_f1:>12.3f}{isaret}')

np.save(f'{DRIVE_DIR}/best_thresholds_v4.npy', best_thresholds)

preds_old = (val_probs > 0.5).astype(int)
preds_new = (val_probs > best_thresholds).astype(int)
print(f'\n=== VAL TOPLAM ETKİ ===')
print(f'F1_macro (0.5):        {f1_score(val_labels, preds_old, average="macro", zero_division=0):.3f}')
print(f'F1_macro (özel eşik):  {f1_score(val_labels, preds_new, average="macro", zero_division=0):.3f}')
print(f'\n✓ best_thresholds_v4.npy kaydedildi')

In [ ]:
# === TEST SETİ DEĞERLENDİRMESİ ===

print('Test seti üzerinde tahmin yapılıyor...')
test_probs, test_labels = get_probs_and_labels(model, test_loader)

preds_05     = (test_probs > 0.5).astype(int)
preds_custom = (test_probs > best_thresholds).astype(int)

f1_macro_05     = f1_score(test_labels, preds_05,     average='macro', zero_division=0)
f1_macro_custom = f1_score(test_labels, preds_custom, average='macro', zero_division=0)
f1_micro_05     = f1_score(test_labels, preds_05,     average='micro', zero_division=0)
f1_micro_custom = f1_score(test_labels, preds_custom, average='micro', zero_division=0)

print(f'\n=== TEST SETİ SONUÇLARI (V4 EfficientNet-B2) ===')
print(f'                 {"Eşik 0.5":>12} {"Özel eşik":>12}')
print(f'F1_macro        {f1_macro_05:>12.3f} {f1_macro_custom:>12.3f}')
print(f'F1_micro        {f1_micro_05:>12.3f} {f1_micro_custom:>12.3f}')

print(f'\n=== PER-CLASS RAPOR (özel eşikle) ===')
report = classification_report(
    test_labels, preds_custom,
    target_names=[c.replace('Tur_', '') for c in label_cols],
    zero_division=0, digits=3
)
print(report)

# V3 ile karşılaştırma
print('=== V3 vs V4 KARŞILAŞTIRMA ===')
print(f'  V3 F1_macro (özel eşik): 0.306')
print(f'  V4 F1_macro (özel eşik): {f1_macro_custom:.3f}')
print(f'  Fark: {f1_macro_custom - 0.306:+.3f}')

# Sonuçları kaydet
sonuc = {
    'model_version':             'v4_efficientnet_b2',
    'best_epoch':                int(ckpt['epoch']),
    'best_val_f1_macro':         float(ckpt['val_f1_macro']),
    'test_f1_macro_05':          float(f1_macro_05),
    'test_f1_macro_custom':      float(f1_macro_custom),
    'test_f1_micro_05':          float(f1_micro_05),
    'test_f1_micro_custom':      float(f1_micro_custom),
    'thresholds':                {c.replace('Tur_', ''): float(t) for c, t in zip(label_cols, best_thresholds)},
    'train_size': len(df_train), 'val_size': len(df_val), 'test_size': len(df_test)
}
with open(f'{DRIVE_DIR}/training_summary_v4.json', 'w', encoding='utf-8') as f:
    json.dump(sonuc, f, indent=2, ensure_ascii=False)
print('\n✓ training_summary_v4.json kaydedildi')

print(f'\n=== HEDEF KONTROLÜ ===')
checks = [
    ('Test F1_macro >= 0.40 (özel eşik)', f1_macro_custom >= 0.40),
    ('Test F1_micro >= 0.55 (özel eşik)', f1_micro_custom >= 0.55),
    ('V3\'den daha iyi F1_macro',          f1_macro_custom > 0.306),
]
for name, passed in checks:
    print(f'  {"✓" if passed else "✗"} {name}')

In [ ]:
# === R2'YE YÜKLE ===

s3 = boto3.client('s3', endpoint_url=ENDPOINT_URL,
    aws_access_key_id=ACCESS_KEY, aws_secret_access_key=SECRET_KEY,
    region_name='auto')

dosyalar = [
    (f'{DRIVE_DIR}/genre_vision_v4_best.pth',  'model/genre_vision_v4_best.pth'),
    (f'{DRIVE_DIR}/best_thresholds_v4.npy',    'model/best_thresholds_v4.npy'),
    (f'{DRIVE_DIR}/training_summary_v4.json',  'model/training_summary_v4.json'),
]

for yerel, r2_key in dosyalar:
    s3.upload_file(yerel, BUCKET_NAME, r2_key)
    boyut = os.path.getsize(yerel) / 1024 / 1024
    print(f'✓ {r2_key} yüklendi ({boyut:.1f} MB)')

# Doğrulama
print("\nR2'deki model dosyaları:")
response = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix='model/')
for obj in response.get('Contents', []):
    print(f'  {obj["Key"]} ({obj["Size"]/1024/1024:.2f} MB)')

## main.py Güncelleme Notu

V4 modeli deploy etmek için `main.py`'deki model yükleme kısmını güncelle:

```python
# Eski (V3 - ResNet18)
from torchvision import models
base = models.resnet18(weights=None)
base.fc = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(base.fc.in_features, len(label_cols)))

# Yeni (V4 - EfficientNet-B2)
from torchvision.models import efficientnet_b2
base = efficientnet_b2(weights=None)
base.classifier = nn.Sequential(nn.Dropout(p=0.4), nn.Linear(1408, len(label_cols)))
```

Ve dosya yollarını:
```python
MODEL_PATH      = 'genre_vision_v4_best.pth'
THRESHOLDS_PATH = 'best_thresholds_v4.npy'
```